## ClaudeSDKClient


Claude Agent SDK 会自动把会话历史落盘到 ~/.claude/projects/XXX/*.jsonl。 目录结构示例：
![session storage](./imgs/09_claude_sdk_session.png)


client.query 常用的2种用法：
| 使用方式                                                     | 适合场景                             |
| ------------------------------------------------------------ | ------------------------------------ |
| client.query(..., options=ClaudeAgentOptions(resume=sid))    | 进程重启后精确恢复任意历史会话       |
| client.query(..., options=ClaudeAgentOptions(resume=sid, fork_session=True)) | 恢复历史会话，并派生分支，原会话不动 |



## 示例代码

### 模式1：Fresh —— 从头开始的一次性分析

In [ ]:
async def run_fresh(prompt: str) -> str:
    """从头开始一次分析，跑完把 session_id 吐出来，方便后续 resume / fork。"""
    options = ClaudeAgentOptions(
        system_prompt=SYSTEM_PROMPT,
        include_partial_messages=True,
        mcp_servers={"websearch": websearch_server},
        allowed_tools=[
            "Read", "Grep", "Glob", "Agent", "AskUserQuestion",
            "mcp__websearch__tavilysearch",
        ],
        agents=build_agents(),
    )

    session_id = None
    async with ClaudeSDKClient(options=options) as client:
        await client.query(prompt)
        async for msg in client.receive_response():
            print(msg)
            if isinstance(msg, ResultMessage):
                session_id = msg.session_id
    return session_id

session_id 是 ResultMessage 的一个字段(ResultMessage 示例见 `08_subagent/helloworld/helloworld_terminal.log` 最后一行),即，任务完成后 获取生成的 session_id.

注：所有message中都有session_id字段

### resume
接着上次聊

In [ ]:
async def run_resume(session_id: str, follow_up: str) -> None:
    """基于历史的 session_id 继续追问。SubAgent 状态、历史消息、Skills 缓存全部复用。"""
    options = ClaudeAgentOptions(
        resume=session_id,
        allowed_tools=[
            "Read", "Grep", "Glob", "Agent", "AskUserQuestion",
            "mcp__websearch__tavilysearch",
        ],
    )
    async with ClaudeSDKClient(options=options) as client:
        await client.query(follow_up)
        async for msg in client.receive_response():
            print(msg)

options中没有重新传 system_prompt 等等，是因为历史会话里已经记录了

### fork
在原会话上派生分支

In [ ]:
async def run_fork(session_id: str, alternative: str) -> str:
    """把当前会话完整复制一份给新分支，分支里随便试错，原会话纹丝不动。"""
    options = ClaudeAgentOptions(
        resume=session_id,
        fork_session=True,           # ★ 关键：派生分支，会把当前历史完整复制一份给新分支，新分支的修改不会回写原会话
        max_turns=5,
    )
    forked_id = None
    async with ClaudeSDKClient(options=options) as client:
        await client.query(alternative)
        async for msg in client.receive_response():
            print(msg)
            if isinstance(msg, ResultMessage):
                forked_id = msg.session_id
    return forked_id

### main()的改造
大意是根据入参判断走哪个 if-else 分支

In [ ]:
async def main():
    parser = argparse.ArgumentParser(
        description="燕京啤酒投研 Agent —— 支持 Fresh / Resume / Fork 三种会话模式"
    )
    parser.add_argument(
        "--resume",
        metavar="SESSION_ID",
        help="从已有 session_id 继续追问",
    )
    parser.add_argument(
        "--fork",
        metavar="SESSION_ID",
        help="基于已有 session_id 派生分支",
    )
    parser.add_argument(
        "prompt",
        nargs="?",
        default=DEFAULT_PROMPT,
        help="要发给主 Agent 的 prompt",
    )
    args = parser.parse_args()

    if args.fork:
        forked = await run_fork(args.fork, args.prompt)
        print(f"\n[FORK_SESSION_ID] {forked}")
        print("# 之后可以继续基于这个分支追问：")
        print(f"# python agent.py --resume {forked} \"在 DCF 估值基础上，把永续增长率调到 1% 重新算一下\"")
    elif args.resume:
        await run_resume(args.resume, args.prompt)
    else:
        sid = await run_fresh(args.prompt)
        print(f"\n[SESSION_ID] {sid}")
        print("# 之后用：")
        print(f"# python agent.py --resume {sid} \"请补充最新一周行业利空\"")
        print("# 或者开分支探索不同投资逻辑：")
        print(f"# python agent.py --fork {sid} \"请基于现有分析，额外用 DCF 模型重做估值\"")

演示：
1. 先运行没有参数的命令(任务同 08_subagent invest)，获取session id
2. 演示 resume。 一周后，市场有了新消息，需要在原分析基础上再做一次增量分析，但不需要重新开始跑。
> $ python agent.py --resume 1f132a87-b9d9-4777-9b64-4a7ceef0adcd "消息：2026 年上半年，啤酒行业销量成整体下滑趋势，7 月随为夏季传统啤酒销售季，但受宏观经济、消费场景 疲软等影响，整体销量同比仍面临一定的下行压力。 请结合这条消息，重点更新风险评估，财报部分沿用上次结论"

3. 演示 fork。 分别使用2种不同的估值方法分析，即开启2个分支
> $ python agent.py --fork 1f132a87-b9d9-4777-9b64-4a7ceef0adcd \
    "请基于现有分析，额外用 DCF 模型重做估值，重点关注 WACC 假设"
> $ python agent.py --fork 1f132a87-b9d9-4777-9b64-4a7ceef0adcd \
    "请基于现有分析，额外用 PE/PB 历史分位法做估值，给出 25%/50%/75% 三个情景"

Fork 让我们可以在多条思路上进行并行探索，互不污染上下文，互不污染结果。

而且还能在分支里继续追问：
> $ python agent.py --resume {forked_session_id} "在 DCF 估值基础上，把永续增长率调到 1% 重新算一下"

即传入 上一轮fork会话结束后的session_id.

分支之上还能再分，SubAgent 的工作记忆被完整保留。